# Course 6 · Databases and SQL for Data Science with Python
We load the practice CSVs into **SQLite** (a whole database in one file, no server needed) and query it two ways: with pandas, and with the `%sql` magic used in the course.
SQL keywords are written in CAPITALS by convention; SQL itself does not care. Comments in SQL start with `--`.

## Step 1 · Create the database

In [ ]:
# os lets Python talk to the operating system (files, folders).
import os
# If the practice data file is not in this folder yet...
if not os.path.exists("applications.csv"):
    # ...run make_data.py to create applications.csv and centers.csv.
    %run make_data.py
# pandas = tables (nickname pd); numpy = fast math (nickname np). These nicknames are standard.
import pandas as pd, numpy as np
import sqlite3                                   # SQLite support is built into Python

con = sqlite3.connect("id_registry.db")          # open (or create) the database file -> a connection
# .to_sql() writes a DataFrame into a database table.
# if_exists="replace" overwrites an old table; index=False skips the pandas row numbers.
pd.read_csv("applications.csv").to_sql("applications", con, if_exists="replace", index=False)
pd.read_csv("centers.csv").to_sql("centers", con, if_exists="replace", index=False)

def q(sql, params=()):
    """Run a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, con, params=params)

# sqlite_master is SQLite's own catalogue: list the tables we just created.
q("SELECT name FROM sqlite_master WHERE type='table'")

## Step 2 · SELECT, WHERE, ORDER BY, LIMIT

In [ ]:
q("""
SELECT app_id, age, channel, processing_days    -- which columns to return
FROM applications                               -- from which table
WHERE docs_complete = 1                         -- keep only rows that pass these tests...
  AND age BETWEEN 18 AND 30                     -- ...BETWEEN includes both ends
ORDER BY processing_days DESC                   -- sort, largest first (ASC = smallest first)
LIMIT 5                                         -- return only the first 5 rows
""")

## Step 3 · DISTINCT, LIKE, IN, NULL

In [ ]:
# DISTINCT: each different value once -> reveals the messy "Walk-In " label.
display(q("SELECT DISTINCT channel FROM applications"))
# LIKE with % = "any characters": '%walk%' matches anything containing "walk".
display(q("SELECT COUNT(*) AS walkin_like FROM applications WHERE LOWER(channel) LIKE '%walk%'"))
# IN (...) = matches any value in the list (shorter than many ORs).
display(q("SELECT region, staff FROM centers WHERE region IN ('Afar', 'Somali')"))
# Missing values are NULL in SQL; test with IS NULL (never "= NULL").
display(q("SELECT COUNT(*) AS missing_age FROM applications WHERE age IS NULL"))

## Step 4 · Aggregate with GROUP BY and HAVING
`WHERE` filters rows *before* grouping; `HAVING` filters groups *after*.

In [ ]:
q("""
SELECT channel,                                         -- the grouping column
       COUNT(*)                                   AS n,          -- rows per group; AS names the result
       ROUND(AVG(processing_days), 1)             AS avg_days,   -- average, rounded to 1 decimal
       ROUND(100.0 * SUM(rejected) / COUNT(*), 1) AS reject_pct  -- 100.0 forces decimal (not whole-number) division
FROM applications
GROUP BY channel                                        -- one output row per channel
HAVING COUNT(*) > 100                                   -- drop small groups
ORDER BY n DESC
""")

## Step 5 · Fix data with UPDATE and DELETE

In [ ]:
cur = con.cursor()                               # a cursor runs commands that change data
# UPDATE ... SET changes values in rows that match WHERE. TRIM removes spaces; LOWER lowercases.
cur.execute("UPDATE applications SET channel = 'walk-in' WHERE TRIM(LOWER(channel)) = 'walk-in'")
# Remove duplicates: rowid is SQLite's hidden row number. Keep the first row per app_id, delete the rest.
cur.execute("""DELETE FROM applications
               WHERE rowid NOT IN (SELECT MIN(rowid) FROM applications GROUP BY app_id)""")
con.commit()                                     # save the changes permanently
q("SELECT channel, COUNT(*) AS n FROM applications GROUP BY channel")   # check: 3 clean labels

## Step 6 · JOINs: combining tables

In [ ]:
# INNER JOIN: only rows with a match in BOTH tables.
q("""
SELECT c.region,                                   -- c. and a. say which table a column comes from
       COUNT(a.app_id)                  AS apps,
       ROUND(AVG(a.processing_days), 1) AS avg_days
FROM applications AS a                             -- "a" is a short alias for applications
JOIN centers      AS c ON a.center_id = c.center_id   -- match rows where center_id is equal
GROUP BY c.region
ORDER BY avg_days DESC
""")

In [ ]:
# LEFT JOIN: keeps EVERY row of the left table (centers), even with no matching applications.
# A center with no applications would show apps = 0 here, but disappear in an INNER JOIN.
q("""
SELECT c.center_id, c.region, COUNT(a.app_id) AS apps
FROM centers AS c
LEFT JOIN applications AS a ON a.center_id = c.center_id
GROUP BY c.center_id
ORDER BY apps
LIMIT 5
""")

## Step 7 · Subqueries: a query inside a query

In [ ]:
q("""
SELECT center_id, ROUND(AVG(processing_days), 2) AS avg_days
FROM applications
GROUP BY center_id
-- The inner (SELECT ...) runs first and returns one number: the overall average.
HAVING AVG(processing_days) > (SELECT AVG(processing_days) FROM applications)
ORDER BY avg_days DESC
""").head()                                        # .head() = first 5 rows of the result

## Step 8 · Parameters (safe from SQL injection)
Never build SQL with f-strings from user input. Use `?` placeholders; values are passed separately and cannot change the query's meaning.

In [ ]:
# Each ? is filled in order from the tuple: region = 'Oromia', staff >= 10.
q("SELECT * FROM centers WHERE region = ? AND staff >= ?", ("Oromia", 10))

## Step 9 · Bonus: window functions (not in the course, common in interviews)

In [ ]:
q("""
SELECT region, center_id, staff,
       -- RANK() numbers rows 1,2,3...; OVER(...) says how:
       -- PARTITION BY region = restart numbering in each region; ORDER BY staff DESC = most staff gets 1.
       RANK() OVER (PARTITION BY region ORDER BY staff DESC) AS staff_rank
FROM centers
ORDER BY region, staff_rank
""").head(10)

## Step 10 · The `%sql` magic (the course's way)
The course used `ipython-sql`; its maintained successor `jupysql` (installed in 00_setup) uses the same commands.

In [ ]:
# Load the SQL extension (adds the %sql and %%sql magics).
%load_ext sql
# Connect to our SQLite file. sqlite:/// + file name is the connection address.
%sql sqlite:///id_registry.db

In [ ]:
%%sql
-- %%sql on the first line turns the whole cell into SQL; the result shows as a table.
SELECT region, COUNT(*) AS centers, SUM(staff) AS total_staff
FROM centers
GROUP BY region
ORDER BY total_staff DESC

## ✍️ Your turn
1. Top 3 centers by rejection rate (minimum 100 applications).
2. Average processing days per month (`strftime('%m', submitted)` extracts the month).
3. For each region, the share of applications that came by mobile unit.

In [ ]:
# your answer here